# Tabular MLP for TDE Classification

**Conditional Tuning Architecture - config-driven tune/fixed params**

3-layer MLP with PReLU, BatchNorm, and Focal Loss.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config_loader import get_path, get_seed, get

RANDOM_STATE = get_seed()
MODEL_NAME = 'mlp'
TUNE_MODE = get('models.mlp.tune')
N_OPTUNA_TRIALS = get('models.mlp.optuna_trials')
common = get('models.mlp.common_params')
N_EPOCHS_TUNING = common.get('n_epochs_tuning', 30)
N_EPOCHS_FINAL = common.get('n_epochs_final', 70)
N_BAGS = common.get('n_bags', 5)
BATCH_SIZE = common.get('batch_size', 128)

DATA_DIR = get_path('data_processed')
MODEL_DIR = get_path('models')
SUBMISSION_DIR = get_path('submissions')
TRAIN_PATH = DATA_DIR / 'further_train_processed_nn.parquet'
TEST_PATH = DATA_DIR / 'further_test_processed_nn.parquet'
FOLDS_PATH = DATA_DIR / 'train_folds.csv'
USE_SELECTED_FEATURES = False
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
print(f'Tune Mode: {TUNE_MODE}, Optuna Trials: {N_OPTUNA_TRIALS}')

In [ ]:
import json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import optuna
from sklearn.metrics import precision_recall_curve
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1-pt)**self.gamma * bce).mean()

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)
train = train.merge(folds[['object_id', 'kfold']], on='object_id', how='left')
drop_cols = get('meta_columns')
feature_cols = [c for c in train.columns if c not in drop_cols]
X = train[feature_cols].values.astype(np.float32)
y = train['target'].values.astype(np.float32)
kfold = train['kfold'].values
X_test = test[feature_cols].values.astype(np.float32)
object_ids_train = train['object_id']
object_ids_test = test['object_id']
INPUT_DIM = len(feature_cols)

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropouts):
        super().__init__()
        layers = []
        prev = input_dim
        for hid, drop in zip(hidden_dims, dropouts):
            layers.extend([nn.Linear(prev, hid), nn.BatchNorm1d(hid), nn.PReLU(), nn.Dropout(drop)])
            prev = hid
        self.layers = nn.Sequential(*layers)
        self.out = nn.Linear(prev, 1)
    def forward(self, x): return self.out(self.layers(x)).squeeze(-1)

In [ ]:
def train_epoch(model, loader, crit, opt, sched=None):
    model.train()
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward(); opt.step()
        if sched: sched.step()
def predict(model, X, bs=512):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            preds.extend(torch.sigmoid(model(torch.FloatTensor(X[i:i+bs]).to(device))).cpu().numpy())
    return np.array(preds)

In [ ]:
def objective(trial):
    tp = get('models.mlp.tune_params')
    h1 = trial.suggest_categorical('hidden1', tp['hidden1'])
    h2 = trial.suggest_categorical('hidden2', tp['hidden2'])
    h3 = trial.suggest_categorical('hidden3', tp['hidden3'])
    d1 = trial.suggest_float('dropout1', *tp['dropout1'])
    d2 = trial.suggest_float('dropout2', *tp['dropout2'])
    d3 = trial.suggest_float('dropout3', *tp['dropout3'])
    lr = trial.suggest_float('lr', tp['lr'][0], tp['lr'][1], log=True)
    wd = trial.suggest_float('weight_decay', tp['weight_decay'][0], tp['weight_decay'][1], log=True)
    gamma = trial.suggest_float('gamma', *tp['gamma'])
    hidden_dims, dropouts = [h1, h2, h3], [d1, d2, d3]
    oof = np.zeros(len(y))
    for fold in range(5):
        tr_idx, val_idx = kfold != fold, kfold == fold
        torch.manual_seed(RANDOM_STATE + fold)
        loader = DataLoader(TensorDataset(torch.FloatTensor(X[tr_idx]), torch.FloatTensor(y[tr_idx])), batch_size=BATCH_SIZE, shuffle=True)
        model = TabularMLP(INPUT_DIM, hidden_dims, dropouts).to(device)
        crit = FocalLoss(gamma=gamma)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_EPOCHS_TUNING * len(loader), 1e-6)
        for _ in range(N_EPOCHS_TUNING): train_epoch(model, loader, crit, opt, sched)
        oof[val_idx] = predict(model, X[val_idx])
    prec, rec, _ = precision_recall_curve(y, oof)
    return np.max(2*prec*rec/(prec+rec+1e-9))

In [ ]:
if TUNE_MODE:
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_params = study.best_params.copy()
    print(f'Best F1: {study.best_value:.4f}')
    print(f'Best params: {best_params}')
else:
    best_params = get('models.mlp.fixed_params').copy()
    print('Using fixed params')

In [ ]:
hidden_dims = [best_params['hidden1'], best_params['hidden2'], best_params['hidden3']]
dropouts = [best_params['dropout1'], best_params['dropout2'], best_params['dropout3']]
oof_preds, test_preds = np.zeros(len(y)), np.zeros(len(X_test))
for fold in range(5):
    tr_idx, val_idx = kfold != fold, kfold == fold
    fold_val, fold_test = np.zeros(val_idx.sum()), np.zeros(len(X_test))
    loader = DataLoader(TensorDataset(torch.FloatTensor(X[tr_idx]), torch.FloatTensor(y[tr_idx])), batch_size=BATCH_SIZE, shuffle=True)
    for bag in range(N_BAGS):
        torch.manual_seed(RANDOM_STATE + bag*100)
        model = TabularMLP(INPUT_DIM, hidden_dims, dropouts).to(device)
        crit = FocalLoss(gamma=best_params['gamma'])
        opt = torch.optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_EPOCHS_FINAL * len(loader), 1e-6)
        for _ in range(N_EPOCHS_FINAL): train_epoch(model, loader, crit, opt, sched)
        fold_val += predict(model, X[val_idx]) / N_BAGS
        fold_test += predict(model, X_test) / N_BAGS
    oof_preds[val_idx] = fold_val
    test_preds += fold_test / 5
prec, rec, thresh = precision_recall_curve(y, oof_preds)
f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
best_thresh = thresh[np.argmax(f1)]
print(f'OOF F1: {np.max(f1):.4f} @ {best_thresh:.4f}')